# 03 — Plani i preprocessing-ut

**Autor:** Jeta Syla  
**Versioni:** 1.0  
**Data:** 11.10.2026  
**Statusi:** Përditësuar më 20.10.2026 sipas zgjedhjes së nënbashkësisë.

## Qëllimi

Qëllimi i këtij plani është të përcaktojë cilat kolona do të mbahen,
largohen ose krijohen dhe cilat metoda do të përdoren për përgatitjen
e datasetit për analizën e biodiversitetit në Kosovë.

## Plani i preprocessing-ut

| Requirement | Columns | Method | Reason |
|---|---|---|---|
| Reduktimi i dimensionit | `iucnRedListStatus` | Largohet sepse është e njëjta kategori IUCN e shkruar me fjalë. `taxonKey` dhe `speciesKey` mbahen. PCA nuk përdoret. | Te `reports/redundanca.md` përputhja 1:1 është 100% për 8/8 kode. Çelësat e taksonit nuk janë 1:1. |
| Zgjedhja e nënbashkësisë | `taxonKey`, `speciesKey`, `datasetKey`, `scientificName`, `species`, `kingdom`, `phylum`, `class`, `order`, `family`, `genus`, `taxonRank`, `iucnRedListCategory`, `eventDate`, `year`, `month`, `day`, `decimalLatitude`, `decimalLongitude`, `coordinateUncertaintyInMeters`, `locality`, `municipality`, `district`, `protectedArea`, `protectedAreaDesignation`, `strictlyProtected`, `basisOfRecord`, `institutionCode`, `issue` | Përzgjedhje manuale. 29 kolona mbahen dhe 13 largohen. | Vendimi është te `reports/zgjedhja_e_vetive.md`, sipas rëndësisë, redundancës, mungesave, tipit dhe qëllimit të analizës. Nuk përdoret `SelectKBest`. |
| Krijimi i vetive | `month`, `year`, `iucnRedListCategory`, `protectedArea`, `species`, `district` | `season` krijohet nga `month`; `decade = (year // 10) * 10`; `isThreatened = 1` për `VU`, `EN`, `CR`; `inProtectedArea = 1` kur `protectedArea` ka vlerë; `observationCount` llogaritet sipas `species`, `district` dhe `year`. | Këto veti mundësojnë analizën sezonale, ndryshimet sipas dekadave, identifikimin e specieve të rrezikuara, krahasimin brenda/jashtë zonave të mbrojtura dhe numrin e vrojtimeve. |
| Diskretizimi | `coordinateUncertaintyInMeters`, `year` | `coordinateUncertaintyInMeters`: ≤100 m = e lartë, 101–1000 m = mesatare, >1000 m = e ulët. `year` grupohet në dekada. `elevation` nuk diskretizohet, sepse doli nga nënbashkësia. | Mbeten vetëm kolonat që u mbajtën më 20.10.2026. |
| Binarizimi | `strictlyProtected`, `kingdom`, `basisOfRecord` | `strictlyProtected` konvertohet nga `TRUE/FALSE` në `1/0`. Për `kingdom` dhe `basisOfRecord` përdoret `OneHotEncoder(handle_unknown="ignore")`. `iasUnionConcern` nuk binarizohet, sepse doli nga nënbashkësia. | `kingdom` dhe `basisOfRecord` kanë nga 7 kategori. |
| Transformimi | `eventDate`, `iucnRedListCategory`, `coordinateUncertaintyInMeters`, `decimalLatitude`, `decimalLongitude` | `eventDate` konvertohet me `pd.to_datetime`. `NE` dhe `DD` trajtohen si të pavlerësuara. Për `coordinateUncertaintyInMeters` përdoret `np.log1p`. Koordinatat ruhen të pandryshuara për harta. | `individualCount` dhe `elevation` dolën nga nënbashkësia, prandaj nuk shkallëzohen. |

## Decision + Rationale

**Decision:** Nënbashkësia analitike ka 29 kolona. Largohen 13,
përfshirë `iucnRedListStatus`, `elevation`, `individualCount`,
`iasUnionConcern` dhe `iasAppliesFrom`. `taxonKey` dhe `speciesKey`
mbeten. Nuk përdoret PCA.

**Rationale:** Vendimi i 20.10.2026 zëvendëson listën e 9 tetorit.
Kolona mbetet nëse i përgjigjet një pyetjeje të projektit dhe nuk është
e tepërt. Kolona largohet nëse rëndësia është e ulët ose mungesa kalon
80% pa qenë e nevojshme për pyetjen. Lista e plotë është te
`reports/zgjedhja_e_vetive.md`.

## Dokumentimi i detyrës

- **Input:** `dataset/kosovo_overall_biodiversity.csv` dhe grupet e atributeve në `README.md`.
- **Çka u bë:** Plani i 9 tetorit u përditësua me nënbashkësinë e 20.10.2026.
- **Rreshta para / pas:** 46,831 / 46,831.
- **Kolona para / pas:** 42 në CSV / 29 në nënbashkësinë analitike.
- **Pse u bë:** Për të përcaktuar metodat para zbatimit të preprocessing-ut.
- **Output:** `notebooks/03_vetite.ipynb`.
- **Commit:** `Add initial preprocessing plan`.

## Vetitë kohore

Tabela e muajve dhe vendimi janë te `reports/vetite_kohore.md`.
Funksioni më poshtë i lexon kolonat origjinale `year` dhe `month`.
Nuk i plotëson nga `eventDate` dhe nuk e ndryshon CSV-në.


In [ ]:
import pandas as pd

SEASON_BY_MONTH = {
    1: "dimër",
    2: "dimër",
    3: "pranverë",
    4: "pranverë",
    5: "pranverë",
    6: "verë",
    7: "verë",
    8: "verë",
    9: "vjeshtë",
    10: "vjeshtë",
    11: "vjeshtë",
    12: "dimër",
}

RECENT_FROM_YEAR = 2017


def add_time_features(df):
    out = df.copy()
    month = pd.to_numeric(out["month"], errors="coerce")
    year = pd.to_numeric(out["year"], errors="coerce")

    out["season"] = month.map(SEASON_BY_MONTH)
    out["decade"] = ((year // 10) * 10).astype("Int64")

    is_recent = pd.Series(pd.NA, index=out.index, dtype="boolean")
    known_year = year.notna()
    is_recent.loc[known_year] = year.loc[known_year] >= RECENT_FROM_YEAR
    out["is_recent"] = is_recent
    return out


### Dokumentimi i hapit: Vetitë kohore

1. **Input:** kolonat origjinale `year` dhe `month`, pasi rifreskohen më 19.10.2026.
2. **Çka u bë:** U shkrua `add_time_features()` për `season`, `decade` dhe `is_recent`.
3. **Rreshta para / pas:** 46,831 / 46,831. CSV-ja mbetet me 42 kolona. Funksioni shton tri kolona vetëm te rezultati.
4. **Pse u bë:** Sezoni, dekada dhe freskia dalin nga `month` dhe `year`. Ku burimi mungon, vlera e re mbetet NaN.
5. **Output:** `reports/vetite_kohore.md` dhe funksioni në këtë notebook.
6. **Commit:** `Add season, decade and is_recent from year and month` (mesazh i propozuar).
7. **Decision + Rationale:** Pragu është `year >= 2017`, sepse viti më i ri është 2026. Arsyetimi i plotë dhe tabela e muajve janë te `reports/vetite_kohore.md`.


## Vetitë taksonomike dhe të ruajtjes

Tabelat e mapping-ut dhe vendimi janë te `reports/vetite_taksonomike.md`.
Funksioni më poshtë i lexon `class` dhe `iucnRedListCategory`.
Nuk e ndryshon CSV-në.


In [ ]:
import pandas as pd

CLASS_TO_GROUP = {"Aves": "Zogj", "Insecta": "Insekte"}
for name in (
    "Magnoliopsida", "Liliopsida", "Polypodiopsida", "Pinopsida", "Bryopsida",
    "Lycopodiopsida", "Marchantiopsida", "Chlorophyceae", "Trebouxiophyceae",
    "Charophyceae", "Glaucophyceae",
):
    CLASS_TO_GROUP[name] = "Bimë"
for name in (
    "Lecanoromycetes", "Agaricomycetes", "Eurotiomycetes", "Dothideomycetes",
    "Pezizomycetes", "Leotiomycetes", "Sordariomycetes", "Rhizophydiomycetes",
    "Chytridiomycetes", "Arthoniomycetes", "Candelariomycetes", "Lobulomycetes",
    "Saccharomycetes", "Dacrymycetes", "Cystobasidiomycetes", "Orbiliomycetes",
    "Pucciniomycetes", "Laboulbeniomycetes", "Mesochytriomycetes", "Sareomycetes",
    "Taphrinomycetes", "Tremellomycetes",
):
    CLASS_TO_GROUP[name] = "Kërpudha"
for name in (
    "Arachnida", "Gastropoda", "Mammalia", "Squamata", "Amphibia", "Clitellata",
    "Bivalvia", "Malacostraca", "Testudines", "Diplopoda", "Chilopoda",
    "Petromyzonti", "Branchiopoda", "Chromadorea", "Copepoda", "Eurotatoria",
    "Cephalopoda", "Polychaeta", "Oligohymenophorea", "Chrysophyceae",
    "Bacillariophyceae", "Peronosporea", "Prymnesiophyceae", "Cyrtophoria",
    "Gymnostomatea", "Phytomyxea", "Dictyochophyceae", "Dinophyceae",
    "Litostomatea", "Perkinsea", "Tubothalamea", "Alphaproteobacteria",
    "Euglenoidea",
):
    CLASS_TO_GROUP[name] = "Tjera"

THREATENED = {"VU", "EN", "CR"}
NOT_THREATENED = {"LC", "NT"}


def add_taxon_features(df):
    out = df.copy()
    klass = out["class"]
    group = klass.map(CLASS_TO_GROUP)
    group = group.where(klass.notna(), "E panjohur")
    out["taxon_group"] = group.fillna("Tjera")

    category = out["iucnRedListCategory"]
    is_threatened = pd.Series(pd.NA, index=out.index, dtype="Int64")
    is_threatened.loc[category.isin(THREATENED)] = 1
    is_threatened.loc[category.isin(NOT_THREATENED)] = 0
    out["is_threatened"] = is_threatened
    return out


### Dokumentimi i hapit: Vetitë taksonomike dhe të ruajtjes

1. **Input:** `class` dhe `iucnRedListCategory`, sipas zgjedhjes së vetive të 13.10.2026.
2. **Çka u bë:** U shkrua `add_taxon_features()` për `taxon_group` dhe `is_threatened`. `EX` u la NaN dhe u dokumentua veçmas.
3. **Rreshta para / pas:** 46,831 / 46,831. CSV-ja mbetet me 42 kolona. Funksioni shton dy kolona vetëm te rezultati.
4. **Pse u bë:** Grupi biologjik dhe rrezikimi dalin nga `class` dhe kategoria IUCN. `NE`, `DD`, NaN dhe `EX` nuk bëhen 0.
5. **Output:** `reports/vetite_taksonomike.md` dhe funksioni në këtë notebook.
6. **Commit:** `Add taxon_group and is_threatened from class and IUCN category` (mesazh i propozuar).
7. **Decision + Rationale:** `VU`, `EN`, `CR` janë 1. `LC` dhe `NT` janë 0. `class` që mungon është `E panjohur`. Arsyetimi i plotë është te `reports/vetite_taksonomike.md`.
